# UC2 — 02 HMM Training

Fits a `CategoricalHMM` over the symbol sequences produced by notebook 01. Searches the grid `n_components ∈ {7, 9, 11} × seeds ∈ {0..7}` in parallel using a `fork` multiprocessing context on two-thirds of available CPU cores, and selects the fit with the minimum BIC.


## 1 · Setup

In [1]:
import sys, pickle
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd

from uc2_hmm_utils import train_multi
from uc2_symbols   import N_SYMBOLS, SYMBOL_NAMES

OUT = PROJECT_ROOT / 'outputs'

## 2 · Load sequences from notebook 01

In [2]:
z = np.load(OUT / 'sequences.npz', allow_pickle=True)
account_ids  = z['account_ids']
concatenated = z['concatenated']
lengths      = z['lengths']
print(f'riders: {len(account_ids):,}   symbols: {concatenated.size:,}')

riders: 98,241   symbols: 1,767,777


## 3 · Parallel multi-state × multi-seed fit

In [3]:
result = train_multi(
    X=concatenated,
    lengths=lengths,
    n_features=N_SYMBOLS,
    state_grid=(7, 9, 11),
    n_seeds=8,
    n_iter=50,
)

df = pd.DataFrame([r.__dict__ | {'bic': r.bic, 'aic': r.aic} for r in result['all_results']])
df.sort_values('bic').head(12)

,n_components,seed,log_likelihood,n_params,n_obs,bic,aic
15,9,6,-2.640637e+06,134,1767777,5.283201e+06,5.281542e+06
21,11,5,-2.655832e+06,186,1767777,5.314340e+06,5.312036e+06
22,11,6,-2.666233e+06,186,1767777,5.335142e+06,5.332839e+06
5,7,0,-2.670857e+06,90,1767777,5.343008e+06,5.341894e+06
13,9,5,-2.670590e+06,134,1767777,5.343107e+06,5.341447e+06
19,11,2,-2.670980e+06,186,1767777,5.344636e+06,5.342332e+06
2,7,1,-2.676903e+06,90,1767777,5.355100e+06,5.353985e+06
14,9,7,-2.680490e+06,134,1767777,5.362907e+06,5.361247e+06
17,11,1,-2.683122e+06,186,1767777,5.368920e+06,5.366617e+06
23,11,7,-2.683961e+06,186,1767777,5.370597e+06,5.368294e+06


## 4 · Selected model

In [7]:
best = result['best_result']
model = result['best_model']
print(f'Selected: {best.n_components} states, seed={best.seed}')
print(f'log-likelihood: {best.log_likelihood:,.1f}')
print(f'BIC: {best.bic:,.1f}')

Selected: 9 states, seed=6
log-likelihood: -2,640,636.8
BIC: 5,283,201.2


## 5 · Emission inspection

For each state, show the top-3 symbols it emits. This is what we use in notebook 03 to label states as high-risk vs low-risk.

In [8]:
em = np.asarray(model.emissionprob_)
emit_df = pd.DataFrame(em, columns=[SYMBOL_NAMES[i] for i in range(N_SYMBOLS)])
emit_df.index.name = 'state'

# Top-3 symbol per state
top3 = emit_df.apply(lambda r: ', '.join(r.nlargest(3).index), axis=1)
pd.DataFrame({'top3_symbols': top3})

,top3_symbols
state,
0,"ACTIVATE_FAST_HANDHELD, ACTIVATE_SLOW_HANDHELD..."
1,"OTHER_HANDHELD_PATTERN, ACTIVATE_SLOW_HANDHELD..."
2,"NO_HANDHELD_FOLLOWUP, ACTIVATE_SLOW_HANDHELD, ..."
3,"ACTIVATE_SLOW_HANDHELD, ACTIVATE_FAST_HANDHELD..."
4,"OTHER_HANDHELD_PATTERN, ACTIVATE_SLOW_HANDHELD..."
5,"OTHER_HANDHELD_PATTERN, ACTIVATE_SLOW_HANDHELD..."
6,"NO_HANDHELD_FOLLOWUP, OTHER_HANDHELD_PATTERN, ..."
7,"ACTIVATE_FAST_HANDHELD, ACTIVATE_SLOW_HANDHELD..."
8,"OTHER_HANDHELD_PATTERN, ACTIVATE_SLOW_HANDHELD..."


## 6 · Save model + diagnostics

In [9]:
with open(OUT / 'hmm_best.pkl', 'wb') as f:
    pickle.dump({'model': model, 'result': best, 'all_results': result['all_results']}, f)

emit_df.to_csv(OUT / 'hmm_emissions.csv')
df.to_csv     (OUT / 'hmm_grid_results.csv', index=False)

print('saved: hmm_best.pkl, hmm_emissions.csv, hmm_grid_results.csv')

saved: hmm_best.pkl, hmm_emissions.csv, hmm_grid_results.csv
